In [ ]:
import pandas as pd

df = pd.read_csv("/content/sessionized_browsing_history.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

In [ ]:
df = df.sort_values(
    ["session_id", "timestamp"]
).reset_index(drop=True)

In [ ]:
##category sequences for each session

sessions = []

for session_id, group in df.groupby("session_id"):
    categories = group["category"].tolist()

    if len(categories) >= 2:
        sessions.append(categories)

In [ ]:
len(sessions)

17

In [ ]:
##category names into numbers

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

all_categories = df["category"].dropna().unique()
encoder.fit(all_categories)

LabelEncoder()

In [ ]:
encoder.classes_

array(['entertainment', 'learning', 'productivity', 'researching',
       'search', 'shopping'], dtype=object)

In [ ]:
##encoding for each sessions

encoded_sessions = []

for session in sessions:
    encoded_session = encoder.transform(session)
    encoded_sessions.append(encoded_session)

print(encoded_sessions[0])

[4 0 0 4 1 1 1 1 1 1 1 1 2 2 2 2 4 4 5 0 0 0 0]


In [ ]:
import numpy as np


In [ ]:
##input sequence

sequence_length = 3

X = []
y = []

for session in encoded_sessions:

    for i in range(sequence_length, len(session)):
        input_sequence = session[i-sequence_length:i]
        next_category = session[i]

        X.append(input_sequence)
        y.append(next_category)

X = np.array(X)
y = np.array(y)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (350, 3)
Target shape: (350,)


In [ ]:
##train test data split
##first 80 percent for training

split_index = int(len(encoded_sessions) * 0.8)

train_sessions = encoded_sessions[:split_index]
test_sessions = encoded_sessions[split_index:]

In [ ]:
def create_sequences(session_list, sequence_length):
    X_data = []
    y_data = []

    for session in session_list:
        for i in range(sequence_length, len(session)):
            X_data.append(session[i-sequence_length:i])
            y_data.append(session[i])

    return np.array(X_data), np.array(y_data)


X_train, y_train = create_sequences(
    train_sessions, sequence_length
)

X_test, y_test = create_sequences(
    test_sessions, sequence_length
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

Training examples: 315
Testing examples: 35


In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

number_of_categories = len(encoder.classes_)

model = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=number_of_categories,
        output_dim=16
    ),

    LSTM(32),

    Dense(number_of_categories, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 3, 16)          │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,566 (25.65 KB)

 Trainable params: 6,566 (25.65 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6984 - loss: 1.6271 - val_accuracy: 0.0000e+00 - val_loss: 1.7158
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6984 - loss: 0.9516 - val_accuracy: 0.0000e+00 - val_loss: 1.5947
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8452 - loss: 0.7269 - val_accuracy: 0.9365 - val_loss: 1.2945
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8532 - loss: 0.6458 - val_accuracy: 0.9524 - val_loss: 0.7613
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8571 - loss: 0.5537 - val_accuracy: 0.9524 - val_loss: 0.3449
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8532 - loss: 0.5094 - val_accuracy: 0.9524 - val_loss: 0.2368
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8532 - loss: 0.4929 - val_accuracy: 0.9524 - val_loss: 0.2217
Epoch 8/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8532 - loss: 0.4834 - val_accuracy: 0.9524 - 

In [ ]:
from sklearn.metrics import classification_report, accuracy_score,confusion_matrix,f1_score

In [ ]:
probabilities = model.predict(X_test)

y_pred = np.argmax(probabilities, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
)

print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(number_of_categories),
        target_names=encoder.classes_,
        zero_division=0
    )
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Accuracy: 0.5428571428571428
Macro F1: 0.40540226681531033
               precision    recall  f1-score   support

entertainment       0.44      1.00      0.61         7
     learning       0.50      0.80      0.62         5
 productivity       1.00      0.78      0.88         9
  researching       0.25      0.50      0.33         2
       search       0.00      0.00      0.00         6
     shopping       0.00      0.00      0.00         6

     accuracy                           0.54        35
    macro avg       0.36      0.51      0.41        35
 weighted avg       0.43      0.54      0.45        35



In [ ]:
example_session = encoded_sessions[-1]

last_sequence = example_session[-sequence_length:]

input_data = np.array([last_sequence])

prediction = model.predict(input_data)

predicted_id = np.argmax(prediction[0])

predicted_category = encoder.inverse_transform(
    [predicted_id]
)[0]

print("Predicted next category:", predicted_category)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
Predicted next category: productivity
